In [12]:
# VERSION 2 - STRONGER CATBOOST PIPELINE

import pandas as pd
import numpy as np
import pygeohash as pgh

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# ==========================================
# LOAD DATA
# ==========================================

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

# ==========================================
# TIME FEATURES
# ==========================================

train[['hour', 'minute']] = (
    train['timestamp']
    .str.split(':', expand=True)
    .astype(int)
)

test[['hour', 'minute']] = (
    test['timestamp']
    .str.split(':', expand=True)
    .astype(int)
)

# cyclic hour
train["hour_sin"] = np.sin(2*np.pi*train["hour"]/24)
train["hour_cos"] = np.cos(2*np.pi*train["hour"]/24)

test["hour_sin"] = np.sin(2*np.pi*test["hour"]/24)
test["hour_cos"] = np.cos(2*np.pi*test["hour"]/24)

# ==========================================
# TRAFFIC PERIOD FEATURES
# ==========================================

for df in [train, test]:
    df["is_morning_rush"] = (
        (df["hour"] >= 7) &
        (df["hour"] <= 9)
    ).astype(int)

    df["is_evening_rush"] = (
        (df["hour"] >= 17) &
        (df["hour"] <= 19)
    ).astype(int)

    df["is_peak"] = (
        df["is_morning_rush"] |
        df["is_evening_rush"]
    ).astype(int)

    df["time_slot"] = (
        df["hour"] * 4 +
        (df["minute"] // 15)
    )

# ==========================================
# MISSING VALUES
# ==========================================

train["RoadType"] = train["RoadType"].fillna("Unknown")
test["RoadType"] = test["RoadType"].fillna("Unknown")

train["Weather"] = train["Weather"].fillna("Unknown")
test["Weather"] = test["Weather"].fillna("Unknown")

temp_median = train["Temperature"].median()

train["Temperature"] = train["Temperature"].fillna(temp_median)
test["Temperature"] = test["Temperature"].fillna(temp_median)

# ==========================================
# GEOHASH FEATURES
# ==========================================

def decode_geohash(g):
    lat, lon = pgh.decode(g)
    return pd.Series([float(lat), float(lon)])

train[["lat", "lon"]] = train["geohash"].apply(decode_geohash)
test[["lat", "lon"]] = test["geohash"].apply(decode_geohash)

# geohash hierarchy
train["geo4"] = train["geohash"].str[:4]
train["geo5"] = train["geohash"].str[:5]

test["geo4"] = test["geohash"].str[:4]
test["geo5"] = test["geohash"].str[:5]

# geohash + hour
train["geo_hour"] = (
    train["geohash"] + "_" +
    train["hour"].astype(str)
)

test["geo_hour"] = (
    test["geohash"] + "_" +
    test["hour"].astype(str)
)

# ==========================================
# SIMPLE TARGET AGGREGATES
# ==========================================

geo_mean = (
    train.groupby("geohash")["demand"]
    .mean()
    .to_dict()
)

train["geo_mean_demand"] = (
    train["geohash"]
    .map(geo_mean)
)

test["geo_mean_demand"] = (
    test["geohash"]
    .map(geo_mean)
)

global_mean = train["demand"].mean()

test["geo_mean_demand"] = (
    test["geo_mean_demand"]
    .fillna(global_mean)
)

# ==========================================
# REMOVE TIMESTAMP
# ==========================================

train.drop(columns=["timestamp"], inplace=True)
test.drop(columns=["timestamp"], inplace=True)

# ==========================================
# FEATURES
# ==========================================

TARGET = "demand"

FEATURES = [c for c in train.columns if c != TARGET]

X = train[FEATURES]
y = train[TARGET]

X_test = test[FEATURES]

# ==========================================
# CAT FEATURES
# ==========================================

cat_features = [
    "geohash",
    "geo4",
    "geo5",
    "geo_hour",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

# ==========================================
# KFOLD
# ==========================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        loss_function="RMSE",
        eval_metric="R2",
        random_seed=42,
        verbose=500
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    preds = model.predict(X_valid)

    score = r2_score(y_valid, preds)

    scores.append(score)

    print(f"Fold {fold} R2 = {score:.5f}")

print("\nMean CV R2 =", np.mean(scores))

# ==========================================
# FINAL MODEL
# ==========================================

final_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=500
)

final_model.fit(X, y, cat_features=cat_features)

# ==========================================
# FEATURE IMPORTANCE
# ==========================================

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": final_model.get_feature_importance()
})

importance = importance.sort_values("Importance", ascending=False)

print(importance.head(20))

# ==========================================
# SUBMISSION
# ==========================================

predictions = final_model.predict(X_test)

predictions = np.clip(predictions, 0, None)

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": predictions
})

submission.to_csv("../submissions/submission.csv", index=False)

print("\nsubmission.csv generated")

Train Shape: (77299, 11)
Test Shape : (41778, 10)
0:	learn: 0.0504881	test: 0.0510432	best: 0.0510432 (0)	total: 86.9ms	remaining: 4m 20s
500:	learn: 0.9527827	test: 0.9481506	best: 0.9481506 (500)	total: 48.5s	remaining: 4m 1s
1000:	learn: 0.9616149	test: 0.9535605	best: 0.9535605 (1000)	total: 1m 38s	remaining: 3m 16s
1500:	learn: 0.9659403	test: 0.9556366	best: 0.9556387 (1499)	total: 2m 26s	remaining: 2m 26s
2000:	learn: 0.9689376	test: 0.9567891	best: 0.9567899 (1999)	total: 3m 12s	remaining: 1m 35s
2500:	learn: 0.9712928	test: 0.9575503	best: 0.9575536 (2499)	total: 4m 3s	remaining: 48.6s
2999:	learn: 0.9731117	test: 0.9581050	best: 0.9581052 (2998)	total: 4m 51s	remaining: 0us

bestTest = 0.9581051505
bestIteration = 2998

Shrink model to first 2999 iterations.
Fold 1 R2 = 0.95811
0:	learn: 0.0504314	test: 0.0501790	best: 0.0501790 (0)	total: 83.6ms	remaining: 4m 10s
500:	learn: 0.9524867	test: 0.9490199	best: 0.9490199 (500)	total: 50.2s	remaining: 4m 10s
1000:	learn: 0.9612385